In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from loguru import logger
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# グラフの状態
class OverAllState(TypedDict):
    joke: str
    topic: str
    feedback: str
    funny_or_not: str

# 構造化出力用の Schema を定義し、ジョークの評価基準とする
class Feedback(BaseModel):
    grade: Literal["面白い", "面白くない"] = Field(
        description="このジョークが面白いかどうかを判定する。",
    )
    feedback: str = Field(
        description="ジョークが面白くない場合は、具体的な改善案を提示する。",
    )

# 大規模言語モデルに構造化出力機能を追加し、評価器を定義する
evaluator = model.with_structured_output(Feedback)

# ノード
def model_call_generator(state: OverAllState) -> OverAllState:
    """大規模言語モデルがジョークを生成する"""

    if state.get("feedback"):
        msg = model.invoke(
            f"""
            「{state['topic']}」についてのジョークを書いてください。

            以下の改善案を参考にしてください：
            {state['feedback']}
            """
        )
    else:
        msg = model.invoke(f"「{state['topic']}」についてのジョークを書いてください")

    return {"joke": msg.content}


def model_call_evaluator(state: OverAllState) -> OverAllState:
    """大規模言語モデルがジョークを評価する"""

    grade = evaluator.invoke(
        f"""
        以下のジョークが面白いかどうかを評価してください：

        {state['joke']}
        """
    )
    logger.info(grade)
    return {
        "funny_or_not":grade,
        "feedback": grade.feedback,
    }


# 条件付きエッジ関数：評価結果に基づいてフローを終了するか、ジョーク生成ノードに戻るかを決定する
def route_joke(state: OverAllState) -> Literal["accept", "reject_and_feedback", END]:
    """評価結果に基づいてジョークを採用するか、フィードバックに基づいて再生成するかを決定する"""

    if state["funny_or_not"] == "面白い":
        return "accept"
    elif state["funny_or_not"] == "面白くない":
        return "reject_and_feedback"
    return END

# ワークフローを構築
builder = StateGraph(state_schema=OverAllState)

# ノードを追加
builder.add_node("model_call_generator", model_call_generator)
builder.add_node("model_call_evaluator", model_call_evaluator)

# エッジを追加して各ノードを接続
builder.add_edge(START, "model_call_generator")
builder.add_edge("model_call_generator", "model_call_evaluator")
builder.add_conditional_edges(
    "model_call_evaluator",
    route_joke,
    {
        # route_joke が返す名前：次に実行するノード
        "accept": END,
        "reject_and_feedback": "model_call_generator",
    },
)

# ワークフローをコンパイル
graph = builder.compile()

# ワークフローを呼び出す
try:
    state = graph.invoke({"topic": "猫"},config={"recursion_limit":5})
except Exception as e:
    logger.info("スーパーステップ数が上限に達しました")

print(state["joke"])

# ワークフローグラフを表示
from IPython.display import display
display(graph)